In [71]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import re 
from tqdm import tqdm 
from rapidfuzz import fuzz, process
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from datetime import datetime
from collections import Counter
from collections import defaultdict
from itertools import product
import os 

In [30]:
def clean_name(text):
    if pd.isna(text): 
        return text 
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def set_value(gdf):
    gdf['category'] = None
    gdf['sub_category'] = None
    gdf['type'] = None
    gdf['score'] = 0.0
    gdf['log'] = None
    return gdf

def check_columns(gdf):
    REQUIRED_COLS = {'name', 'subclass', 'review'}

    missing = REQUIRED_COLS - set(gdf.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    else:
        print("✅ All Columns is Clear.")
    
def refine_crs(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    gdf = gdf.to_crs(epsg=3857)

    if gdf.crs is None: 
        print(f' Warning Massage : There is a problem with GDF CRS')

    return gdf

def describe_gdf(gdf):
    print('GDF Description')
    print(f'---------------')

    try: print(f'📦 Total Records : {len(gdf)}')
    except: print('⚠️ Cannot Count GDF Reccord')

    try: print(f'🧱 Columns : {list(gdf.columns)}')
    except: print('⚠️ Cannot Show GDF Reccord')
    
    if gdf.crs : print(f'🌍 CRS : {gdf.crs}') 
    else : print('🌍 CRS : None')

    if 'geometry' in gdf : geom_types = gdf.geom_type.unique()
    else : 'None'
    print(f'🧭 Geometry Type: {geom_types}')

    if 'geometry' in gdf : print(f'📍 Missing geometry: {(gdf.geometry.isna()).sum()}')
    else: print(f'📍 Missing geometry: 0')
    
    print('---------------')
    try:
        gdf = gdf[gdf.geometry.notna()]
        gdf = gdf.to_crs(epsg=4326) if gdf.crs else gdf.set_crs(epsg=4326)
        gdf['lon'] = gdf.geometry.x
        gdf['lat'] = gdf.geometry.y

        print(f"Longitude range: {gdf['lon'].min():.2f} – {gdf['lon'].max():.2f}")
        print(f"Latitude range: {gdf['lat'].min():.2f} – {gdf['lat'].max():.2f}")

        if gdf['lon'].between(95, 141).mean() < 0.95 or gdf['lat'].between(-11, 6).mean() < 0.95:
            print("⚠️ Some coordinates are outside Indonesia bounds.")
        else:
            print("✅ All points within Indonesia bounds.")
    except:
        print('⚠️ Cannot Process Information of Coordinate')
    
    REQUIRED_COLS = {'name', 'subclass', 'review'}

    missing = REQUIRED_COLS - set(gdf.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}, please check and rename the columns')
    else:
        print("✅ All Columns is Clear.")
    

In [34]:
# Configuration

main_df = pd.read_excel('config_engine.xlsx', sheet_name='Main')
operator_df = pd.read_excel('config_engine.xlsx', sheet_name='Operator')
# brand_df = pd.read_excel('config_engine.xlsx', sheet_name='Brand')
# type_df = pd.read_excel('config_engine.xlsx', sheet_name='Type')


In [54]:
config = pd.merge(main_df, operator_df, on=['id', 'sub_category'], how='left')
config

,id,status,category,sub_category,brand,type_by,rule_id,polarity,column,operator,value
0,FS_1,Active,Financial Services,Insurance,No,Review,1.0,Positive,Subclass,Review,"asuransi,kantor perusahaan,bank"
1,FS_1,Active,Financial Services,Insurance,No,Review,1.0,Positive,Name,Review,"asuransi,life,allianz,prudential,insurance,ass..."
2,FS_1,Active,Financial Services,Insurance,No,Review,2.0,Positive,Subclass,Brand,asuransi
3,FS_2,Active,Financial Services,Cooperation,No,Review,1.0,Positive,Name,Startswith,"ksu ,ksp ,kpri ,koperasi,cooperative,kud ,kspp..."
4,FS_3,Active,Financial Services,Pawn Shop,Yes,Brand,1.0,Positive,Subclass,Contain,"gadai,kantor,pinjam,uang,bank,"
5,FS_3,Active,Financial Services,Pawn Shop,Yes,Brand,1.0,Positive,Name,Contain,"pegadaian,pawn,pt pegadaian"
6,FS_3,Active,Financial Services,Pawn Shop,Yes,Brand,2.0,Positive,Name,Startswith,"pegadaian,pawn,pt pegadaian"
7,FB_1,Active,Food and Beverage,Restaurant,No,NaN,NaN,NaN,NaN,NaN,NaN
8,FB_2,Active,Food and Beverage,Fast Food,No,NaN,NaN,NaN,NaN,NaN,NaN
9,FB_3,Active,Food and Beverage,Coffee Shop,No,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
function_map = {}

for _, row in config.iterrows():
    
    }

In [50]:
LIST_NAME = operator_df.loc[3, 'value']
LIST_NAME = LIST_NAME.split(',')
LIST_NAME
# type(LIST_NAME)

['ksu ',
 'ksp ',
 'kpri ',
 'koperasi',
 'cooperative',
 'kud ',
 'kspps ',
 'bmt ',
 'kopma ',
 'kopkar ',
 'kud']

In [ ]:
function_map = {}

for _, row in main_df.iterrows()

In [64]:
# Input Point of Interest (POI) Data

gdf = gpd.read_parquet(rf'C:\Users\Ahmad Haikal\Documents\SPIE 2.0\Data\POI Bali Jawa.parquet')

gdf['cleaned_name'] = gdf['name'].apply(clean_name)
gdf['subclass'] = gdf['subclass'].apply(clean_name) 
gdf = refine_crs(gdf)
gdf = set_value(gdf)

describe_gdf(gdf)

# # Optional Rename for Column Name
# gdf = gdf.rename(columns={
#     'name' : 'name',
#     'subclass' : 'subclass',
#     'review' : 'review'
# })


GDF Description
---------------
📦 Total Records : 27471
🧱 Columns : ['fid', 'name', 'phone', 'subclass', 'address', 'rating', 'review', 'keyword', 'longitude', 'latitude', 'Island', 'Provinsi', 'Kabkot', 'Kecamatan', 'Desa', 'distance', 'poi_id', 'geometry', 'cleaned_name', 'category', 'sub_category', 'type', 'score', 'log']
🌍 CRS : EPSG:3857
🧭 Geometry Type: ['Point']
📍 Missing geometry: 0
---------------
Longitude range: 107.07 – 115.64
Latitude range: -8.84 – -6.26
✅ All points within Indonesia bounds.
✅ All Columns is Clear.


In [77]:
main_df = pd.read_excel('config_engine.xlsx', sheet_name='Main', engine='openpyxl')
category_df = pd.read_excel('config_engine.xlsx', sheet_name='Category', engine='openpyxl')
sub_category_df = pd.read_excel('config_engine.xlsx', sheet_name='Sub Category', engine='openpyxl')
brand_df = pd.read_excel('config_engine.xlsx', sheet_name='Brand', engine='openpyxl')
# type_df = pd.read_excel('config_engine.xlsx', sheet_name='Type', engine='openpyxl')


In [ ]:
main_df[['main_id', 'category']] = main_df[['main_id', 'category']].ffill()
category_df[['main_id', 'category']] = category_df[['main_id', 'category']].ffill()
sub_category_df[['main_id', 'sub_id', 'sub_category']] = sub_category_df[['main_id', 'sub_id', 'sub_category']].ffill()

In [ ]:
def main_registry(main_df):
    registry = {}

    for _, row in tqdm(main_df.iterrows(), total=len(main_df)):
        if row["status"] != "Active":
            continue

        main_id = row["main_id"]
        sub_id = row["sub_id"]

        if main_id not in registry:
            registry[main_id] = {
                "main_id" : main_id,
                "category": row["category"],
                "subcategories": {}
            }

        # init sub category
        registry[main_id]["subcategories"][sub_id] = {
            "name": row["sub_category"], 
            "brand_enabled": row["brand"] == "Yes",
            "type_by": None if row["type_by"] == "No" else row["type_by"]
        }

    return registry

def category_registry(df):
    registry = {}

    for _, row in df.iterrows():
        main_id = row["main_id"]
        category = row["category"]
        list_type = row["list_type"].lower()
        raw_keywords = str(row["keyword"])

        # init category
        if main_id not in registry:
            registry[main_id] = {
                "category" : category,
                "name": {"keyword": [], "noise": []},
                "subclass": {"keyword": [], "noise": []}
            }

        # skip empty
        if raw_keywords.lower() == "nan":
            continue

        keywords = [
            k.lower()
            for k in raw_keywords.split(",")
            if k
        ]

        # mapping list_type → bucket
        if list_type == "name keyword":
            registry[main_id]["name"]["keyword"].extend(keywords)

        elif list_type == "name noise":
            registry[main_id]["name"]["noise"].extend(keywords)

        elif list_type == "subclass keyword":
            registry[main_id]["subclass"]["keyword"].extend(keywords)

        elif list_type == "subclass noise":
            registry[main_id]["subclass"]["noise"].extend(keywords)

    return registry

def sub_category_registry(df):
    registry = {}

    for _, row in df.iterrows():
        main_id = row['main_id']
        sub_id = row['sub_id']
        sub_category = row['sub_category']
        list_type = row["list_type"].lower()
        raw_keywords = str(row["keyword"])

        # init sub category
        if sub_id not in registry:
            registry[sub_id] = {
                "main_id": main_id,
                "sub_category": sub_category,
                "name": {"keyword": [], "noise": []},
                "subclass": {"keyword": [], "noise": []}
            }

        # skip empty
        if raw_keywords.lower() == "nan":
            continue

        keywords = [
            k.strip().lower()
            for k in raw_keywords.split(",")
            if k.strip()
        ]

        # mapping list_type → bucket
        if list_type == "name keyword":
            registry[sub_id]["name"]["keyword"].extend(keywords)

        elif list_type == "name noise":
            registry[sub_id]["name"]["noise"].extend(keywords)

        elif list_type == "subclass keyword":
            registry[sub_id]["subclass"]["keyword"].extend(keywords)

        elif list_type == "subclass noise":
            registry[sub_id]["subclass"]["noise"].extend(keywords)

    return registry


main_map = main_registry(main_df)
category_map = category_registry(category_df)
sub_category_map = sub_category_registry(sub_category_df)

100%|██████████| 42/42 [00:00<00:00, 20608.42it/s]

8
8
37


In [108]:
poi_sample = {
    "name": "Bakso Ayam Syariah",
    "address": "Jl. Jendral Sudirman Jakarta"
}

def evaluate_single_poi(poi, main_map, category_map):
    result = {
        "poi_name": poi["name"],
        "candidates": []
    }

    for main_id, main_cfg in main_map.items():
        result["candidates"].append({
            "main_id": main_id,
            "category": main_cfg["category"],
            "score": 0,
            "reason": []
        })

    return result

evaluate_single_poi(poi_sample, main_map, category_map)

{'poi_name': 'Bakso Ayam Syariah',
 'candidates': [{'main_id': 'FS',
   'category': 'Financial Services',
   'score': 0,
   'reason': []},
  {'main_id': 'FB', 'category': 'Food and Beverage', 'score': 0, 'reason': []},
  {'main_id': 'RS',
   'category': 'Retail and Shopping',
   'score': 0,
   'reason': []},
  {'main_id': 'HE', 'category': 'Higher Education', 'score': 0, 'reason': []},
  {'main_id': 'SC', 'category': 'School', 'score': 0, 'reason': []},
  {'main_id': 'HA',
   'category': 'Hospitality and Accomodation',
   'score': 0,
   'reason': []},
  {'main_id': 'BA', 'category': 'Banking', 'score': 0, 'reason': []},
  {'main_id': 'IN', 'category': 'Industry ', 'score': 0, 'reason': []}]}

In [107]:
def keyword_hit(text, keywords):
    text = text.lower()
    return [k for k in keywords if k in text]

keyword_hit("ATM BCA Sudirman", ["atm", "bca", "mandiri"])

['atm', 'bca']

In [109]:
def evaluate_single_poi(poi, main_map, category_map):
    result = {
        "poi_name": poi["name"],
        "candidates": []
    }

    for main_id, main_cfg in main_map.items():
        score = 0
        reason = []

        cat_cfg = category_map.get(main_id)
        if cat_cfg:
            hits = keyword_hit(
                poi["name"],
                cat_cfg["name"]["keyword"]
            )
            noise = keyword_hit(
                poi["name"],
                cat_cfg["name"]["noise"]
            )

            score += len(hits) * 10
            score -= len(noise) * 10

            if hits:
                reason.append(f"name keyword hit: {hits}")
            if noise:
                reason.append(f"name noise hit: {noise}")

        result["candidates"].append({
            "main_id": main_id,
            "category": main_cfg["category"],
            "score": score,
            "reason": reason
        })

    return result

result = evaluate_single_poi(poi_sample, main_map, category_map)

for r in result["candidates"]:
    print(r)

{'main_id': 'FS', 'category': 'Financial Services', 'score': 0, 'reason': []}
{'main_id': 'FB', 'category': 'Food and Beverage', 'score': 20, 'reason': ["name keyword hit: ['ayam', 'bakso']"]}
{'main_id': 'RS', 'category': 'Retail and Shopping', 'score': 0, 'reason': []}
{'main_id': 'HE', 'category': 'Higher Education', 'score': 0, 'reason': []}
{'main_id': 'SC', 'category': 'School', 'score': 0, 'reason': []}
{'main_id': 'HA', 'category': 'Hospitality and Accomodation', 'score': 0, 'reason': []}
{'main_id': 'BA', 'category': 'Banking', 'score': 0, 'reason': []}
{'main_id': 'IN', 'category': 'Industry ', 'score': 0, 'reason': []}


In [110]:
def pick_best_category(result, min_score=10):
    sorted_candidates = sorted(
        result["candidates"],
        key=lambda x: x["score"],
        reverse=True
    )

    best = sorted_candidates[0]

    if best["score"] < min_score:
        return None

    return best

best_cat = pick_best_category(result)
best_cat

{'main_id': 'FB',
 'category': 'Food and Beverage',
 'score': 20,
 'reason': ["name keyword hit: ['ayam', 'bakso']"]}

In [ ]:
def evaluate_sub_category(poi, best_cat, main_map, sub_category_map):
    main_id = best_cat["main_id"]
    subcats = main_map[main_id]["subcategories"]

    results = []

    for sub_id, sub_cfg in subcats.items():
        score = 0
        reason = []

        sub_kw = sub_category_map.get(sub_id)
        if not sub_kw:
            continue

        hits = keyword_hit(
            poi["name"],
            sub_kw["name"]["keyword"]
        )
        noise = keyword_hit(
            poi["name"],
            sub_kw["name"]["noise"]
        )

        score += len(hits) * 10
        score -= len(noise) * 10

        if hits:
            reason.append(f"sub name hit: {hits}")
        if noise:
            reason.append(f"sub noise hit: {noise}")

        results.append({
            "sub_id": sub_id,
            "sub_category": sub_cfg["name"],
            "score": score,
            "reason": reason
        })

    return results

In [ ]:
text = """
kantor akuntan
konsultan pajak
layanan hukum
firma hukum
jasa akuntansi
konsultan manajemen
akuntan 

"""

result = ",".join(
    line.strip('\n')
    for line in text.splitlines()
    if line.strip()
)

print(result)

kantor akuntan,konsultan pajak,layanan hukum,firma hukum,jasa akuntansi,konsultan manajemen,akuntan 
